# Custom Region Editor

Use this toiol to draw and save **curated custom anatomical regions** on the Allen CCF atlas.

Typical workflow:

1. Enter a custom region name, e.g. `TH-Ant-MM`.
2. Optionally restrict drawing to an existing atlas region, e.g. `TH`.
3. Navigate to an Allen Atlas plate.
4. Click **Draw Polygon**.
5. Click points around the desired region.
6. Use **Undo Point** if needed.
7. Click **Finish Polygon**.
8. Repeat on additional plates.
9. Click **Save Custom Region**.

Saved regions are written to:

- `data/custom_masks/<region>.npz`
- `config/custom_masks.json`

> This editor is intended for region curation. The multiple ROI selector can later load the approved masks.

In [1]:
## 1. Interactive Matplotlib backend
# This notebook needs the `ipympl` interactive backend so mouse clicks on the atlas are captured.

%load_ext autoreload
%autoreload 2

%matplotlib widget

# If `%matplotlib widget` raises an `ipympl` error, install it once with:

# ```python
# %pip install ipympl
# ```

# Then restart the kernel and run the notebook again from the top.

In [ ]:
## 2. Imports
import json
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.path import Path as MplPath

import ipywidgets as widgets
from IPython.display import display, HTML

from ccf_roi_selector.atlas import load_atlas
from ccf_roi_selector.plotting import get_slice
from ccf_roi_selector.roi import (
    load_custom_regions,
    resolve_region_indices,
)
from ccf_roi_selector.paths import (
    ensure_custom_mask_storage,
)

In [ ]:
display(
    HTML(
        """
        <style>
        body {
            background-color: #f8fafc !important;
        }

        .jp-Notebook,
        .jp-OutputArea-output,
        .voila-contents {
            max-width: 1000px;
            margin: 0 auto !important;
        }
        </style>
        """
    )
)

In [ ]:
## 3. Load Allen CCF atlas
template, annotation, parcellation_annotation = load_atlas()

# template.shape, annotation.shape, len(parcellation_annotation)

((456, 320, 528), (456, 320, 528), 3440)

In [ ]:
## 4. Repository paths
user_data_root = ensure_custom_mask_storage()

custom_masks_dir = (
    user_data_root
    / "data"
    / "custom_masks"
)

registry_path = (
    user_data_root
    / "config"
    / "custom_masks.json"
)

print("Custom masks folder:", custom_masks_dir)
print("Registry:", registry_path)

Repository root: c:\Users\maria.vergara\OneDrive - Allen Institute\Documents\Programming\abc-atlas-roi-selector
Custom masks folder: c:\Users\maria.vergara\OneDrive - Allen Institute\Documents\Programming\abc-atlas-roi-selector\data\custom_masks
Custom mask registry: c:\Users\maria.vergara\OneDrive - Allen Institute\Documents\Programming\abc-atlas-roi-selector\config\custom_masks.json


In [ ]:
## 5. Regions available as drawing restrictions

# The custom region can optionally be restricted to an existing Allen region or one of the composite regions defined in `config/custom_regions.json`.

# Example:

# - New region: `THmedial`
# - Restrict to: `TH`

# Any part of the polygon outside `TH` will be discarded.

custom_regions = load_custom_regions()

allen_regions = (
    parcellation_annotation["parcellation_term_acronym"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

base_region_options = sorted(
    set(allen_regions) | set(custom_regions.keys())
)

base_region_options = ["None"] + base_region_options

# print("Available restriction regions:", len(base_region_options))

Available restriction regions: 834


In [6]:
## 6. Allen Atlas plate helpers
def max_plate_from_template(template):
    return ((template.shape[2] - 1) // 4) + 1


def plate_to_slice(plate):
    slice_index = (plate - 1) * 4

    return min(
        slice_index,
        template.shape[2] - 1
    )


def slice_to_plate(slice_index):
    return slice_index // 4 + 1

In [7]:
# ## 7. Editor state

# `custom_mask` is the full 3D boolean mask being curated.

# The manual polygon system stores each mouse click in `current_vertices`.

custom_mask = np.zeros_like(
    annotation,
    dtype=bool
)

drawing_mode = False
current_vertices = []

locked_base_region = None
current_slice_index = None

drawing_cid = None

In [8]:
## 8. Interface widgets

region_name_input = widgets.Text(
    placeholder="e.g. TH-Ant-MM",
    description="Region name:",
    layout=widgets.Layout(width="500px")
)

default_base_region = (
    "TH"
    if "TH" in base_region_options
    else "None"
)

base_region_input = widgets.Combobox(
    options=base_region_options,
    value=default_base_region,
    description="Restrict to:",
    ensure_option=True,
    layout=widgets.Layout(width="500px")
)

max_plate = max_plate_from_template(template)

plate_slider = widgets.IntSlider(
    min=1,
    max=max_plate,
    step=1,
    value=min(53, max_plate),
    description="Atlas Plate:",
    continuous_update=False,
    layout=widgets.Layout(width="700px")
)

prev_button = widgets.Button(
    description="← Previous Plate"
)

next_button = widgets.Button(
    description="Next Plate →"
)

prev_button.style.button_color = "#d9eaf7"
next_button.style.button_color = "#d9eaf7"

draw_button = widgets.Button(
    description="✏ Draw Polygon",
    button_style="info",
    layout=widgets.Layout(width="170px")
)

undo_point_button = widgets.Button(
    description="↶ Undo Point",
    disabled=True,
    layout=widgets.Layout(width="145px")
)

finish_polygon_button = widgets.Button(
    description="✓ Finish Polygon",
    button_style="success",
    disabled=True,
    layout=widgets.Layout(width="175px")
)

cancel_drawing_button = widgets.Button(
    description="Cancel Drawing",
    disabled=True,
    layout=widgets.Layout(width="150px")
)

clear_plate_button = widgets.Button(
    description="Clear Plate",
    layout=widgets.Layout(width="135px")
)

clear_all_button = widgets.Button(
    description="Clear All",
    button_style="danger",
    layout=widgets.Layout(width="115px")
)

save_button = widgets.Button(
    description="💾 Save Custom Region",
    button_style="success",
    layout=widgets.Layout(
        width="220px",
        height="42px"
    )
)

overwrite_checkbox = widgets.Checkbox(
    value=False,
    description="Overwrite if region already exists"
)

status = widgets.HTML()

In [9]:
# ## 9. Create the interactive atlas canvas

# The canvas is inserted directly into the ipywidgets interface so mouse events reach Matplotlib reliably.

with plt.ioff():
    fig, ax = plt.subplots(
        figsize=(8, 7)
    )

fig.canvas.toolbar_visible = True
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.canvas.resizable = True

In [10]:
## 10. Base-region mask for the current plate
def get_base_mask_display():

    base_region = base_region_input.value

    if base_region == "None":
        return None

    indices = resolve_region_indices(
        parcellation_annotation,
        base_region
    )

    annotation_slice = get_slice(
        annotation,
        current_slice_index
    )

    return np.isin(
        annotation_slice,
        indices
    )

In [11]:
# ## 11. Render the current plate

# The figure shows:

# - Allen average template
# - outline of the restriction region
# - previously finished custom-mask polygons
# - the polygon currently being drawn

def render_plate():

    global current_slice_index

    plate = plate_slider.value
    current_slice_index = plate_to_slice(plate)

    template_slice = get_slice(
        template,
        current_slice_index
    )

    saved_mask_slice = get_slice(
        custom_mask,
        current_slice_index
    ).astype(bool)

    ax.clear()

    # Allen average template
    ax.imshow(
        template_slice,
        cmap="gray"
    )

    # Outline of the base/restriction region
    base_mask = get_base_mask_display()

    if (
        base_mask is not None
        and np.any(base_mask)
    ):
        ax.contour(
            base_mask.astype(float),
            levels=[0.5],
            linewidths=1.5
        )

    # Previously accepted mask on this plate
    if np.any(saved_mask_slice):

        visible_mask = np.ma.masked_where(
            ~saved_mask_slice,
            saved_mask_slice
        )

        ax.imshow(
            visible_mask,
            alpha=0.4,
            cmap="autumn",
            interpolation="none"
        )

    # Polygon currently being drawn
    if drawing_mode and current_vertices:

        xs = [point[0] for point in current_vertices]
        ys = [point[1] for point in current_vertices]

        ax.plot(
            xs,
            ys,
            marker="o",
            linewidth=1.5
        )

    ax.set_title(
        f"Allen Atlas Plate {plate}"
    )

    ax.axis("off")

    fig.canvas.draw_idle()

In [12]:
## 12. Convert polygon vertices into a boolean mask

def polygon_to_mask(vertices, shape):

    height, width = shape

    x, y = np.meshgrid(
        np.arange(width),
        np.arange(height)
    )

    points = np.column_stack(
        (
            x.ravel(),
            y.ravel()
        )
    )

    polygon = MplPath(vertices)

    mask = polygon.contains_points(
        points,
        radius=0.5
    )

    return mask.reshape(
        height,
        width
    )

In [13]:
# ## 13. Drawing controls

# When drawing starts:

# - plate navigation is locked
# - each left-click adds one polygon vertex
# - `Undo Point` removes the most recent click
# - `Finish Polygon` closes and commits the polygon
# - `Cancel Drawing` discards the unfinished polygon

def set_drawing_controls(active):

    plate_slider.disabled = active
    prev_button.disabled = active
    next_button.disabled = active

    base_region_input.disabled = (
        active or locked_base_region is not None
    )

    draw_button.disabled = active

    undo_point_button.disabled = not active
    finish_polygon_button.disabled = not active
    cancel_drawing_button.disabled = not active

    clear_plate_button.disabled = active
    clear_all_button.disabled = active
    save_button.disabled = active

In [14]:
def start_drawing(button):

    global drawing_mode
    global current_vertices

    # Make sure pan/zoom is not stealing mouse clicks
    toolbar = fig.canvas.toolbar

    if toolbar.mode == "pan/zoom":
        toolbar.pan()

    elif toolbar.mode == "zoom rect":
        toolbar.zoom()

    drawing_mode = True
    current_vertices = []

    set_drawing_controls(True)

    status.value = (
        "<b>Drawing mode:</b> "
        "left-click around the region. "
        "Use <b>Undo Point</b> if needed, then "
        "<b>Finish Polygon</b>."
    )

    render_plate()


draw_button.on_click(start_drawing)

In [15]:
## 14. Capture mouse clicks
def canvas_clicked(event):

    if not drawing_mode:
        return

    if event.inaxes != ax:
        return

    if event.button != 1:
        return

    # Ignore clicks while a toolbar interaction is active
    if fig.canvas.toolbar.mode:
        return

    if event.xdata is None or event.ydata is None:
        return

    current_vertices.append(
        (
            float(event.xdata),
            float(event.ydata)
        )
    )

    status.value = (
        f"<b>Drawing mode:</b> "
        f"{len(current_vertices)} point(s) selected."
    )

    render_plate()


# Avoid duplicate click callbacks if this cell is re-run
try:
    if drawing_cid is not None:
        fig.canvas.mpl_disconnect(drawing_cid)
except Exception:
    pass

drawing_cid = fig.canvas.mpl_connect(
    "button_press_event",
    canvas_clicked
)

In [16]:
## 15. Undo the last point
def undo_last_point(button):

    if not drawing_mode:
        return

    if not current_vertices:
        return

    current_vertices.pop()

    status.value = (
        f"<b>Drawing mode:</b> "
        f"{len(current_vertices)} point(s) selected."
    )

    render_plate()


undo_point_button.on_click(
    undo_last_point
)

In [17]:
## 16. Cancel an unfinished polygon

def cancel_drawing(button):

    global drawing_mode
    global current_vertices

    drawing_mode = False
    current_vertices = []

    set_drawing_controls(False)

    status.value = (
        "Drawing cancelled."
    )

    render_plate()


cancel_drawing_button.on_click(
    cancel_drawing
)

In [18]:
# ## 17. Finish and commit a polygon

# The polygon is converted into a mask and intersected with the selected restriction region.

# Multiple polygons can be added to the same plate.

def finish_polygon(button):

    global custom_mask
    global locked_base_region
    global drawing_mode
    global current_vertices

    if len(current_vertices) < 3:

        status.value = (
            "<span style='color:red;'>"
            "A polygon needs at least 3 points."
            "</span>"
        )

        return

    display_shape = get_slice(
        template,
        current_slice_index
    ).shape

    drawn_mask = polygon_to_mask(
        current_vertices,
        display_shape
    )

    # Lock the chosen base region after the first accepted polygon
    if locked_base_region is None:
        locked_base_region = base_region_input.value

    # Restrict the polygon to the selected anatomical region
    base_mask = get_base_mask_display()

    if base_mask is not None:
        drawn_mask &= base_mask

    if not np.any(drawn_mask):

        status.value = (
            "<span style='color:red;'>"
            "The polygon does not contain any valid pixels "
            "inside the selected restriction region."
            "</span>"
        )

        return

    # get_slice() rotates the raw array with k=3.
    # k=1 reverses that rotation before storing.
    raw_mask = np.rot90(
        drawn_mask,
        k=1
    )

    # Add instead of replacing, allowing multiple polygons per plate
    custom_mask[
        :,
        :,
        current_slice_index
    ] |= raw_mask

    pixel_count = int(
        drawn_mask.sum()
    )

    drawing_mode = False
    current_vertices = []

    set_drawing_controls(False)

    render_plate()

    status.value = (
        f"<span style='color:green;'>"
        f"✓ Polygon added to Plate {plate_slider.value} "
        f"({pixel_count:,} pixels)."
        f"</span>"
    )


finish_polygon_button.on_click(
    finish_polygon
)

In [19]:
## 18. Plate navigation
def previous_plate(button):

    if plate_slider.value > plate_slider.min:
        plate_slider.value -= 1


def next_plate(button):

    if plate_slider.value < plate_slider.max:
        plate_slider.value += 1


prev_button.on_click(
    previous_plate
)

next_button.on_click(
    next_plate
)


def plate_changed(change):

    if not drawing_mode:
        render_plate()


plate_slider.observe(
    plate_changed,
    names="value"
)

In [20]:
## 19. Clear the current plate
def clear_current_plate(button):

    global locked_base_region

    slice_index = plate_to_slice(
        plate_slider.value
    )

    custom_mask[
        :,
        :,
        slice_index
    ] = False

    # Unlock restriction if the entire custom mask is now empty
    if not np.any(custom_mask):

        locked_base_region = None
        base_region_input.disabled = False

    render_plate()

    status.value = (
        f"Cleared Plate {plate_slider.value}."
    )


clear_plate_button.on_click(
    clear_current_plate
)

In [21]:
## 20. Clear the entire custom region

def clear_all(button):

    global locked_base_region
    global drawing_mode
    global current_vertices

    custom_mask.fill(False)

    locked_base_region = None

    drawing_mode = False
    current_vertices = []

    set_drawing_controls(False)

    base_region_input.disabled = False

    render_plate()

    status.value = (
        "All drawings cleared."
    )


clear_all_button.on_click(
    clear_all
)

In [22]:
# ## 21. Save the custom region

# The editor saves:

# ```text
# data/custom_masks/<name>.npz
# ```

# and records metadata in:

# ```text
# config/custom_masks.json
# ```

# For Windows-compatible filenames, region names may contain letters, numbers, `_`, and `-`.

def save_custom_region(button):

    name = region_name_input.value.strip()

    # Validate region name
    if not name:

        status.value = (
            "<span style='color:red;'>"
            "Enter a region name first."
            "</span>"
        )

        return

    if not re.fullmatch(
        r"[A-Za-z0-9_-]+",
        name
    ):

        status.value = (
            "<span style='color:red;'>"
            "Use only letters, numbers, underscores, or hyphens "
            "in the region name."
            "</span>"
        )

        return

    if not np.any(custom_mask):

        status.value = (
            "<span style='color:red;'>"
            "Draw at least one polygon before saving."
            "</span>"
        )

        return

    mask_path = (
        custom_masks_dir
        / f"{name}.npz"
    )

    # Load existing registry
    if registry_path.exists():

        with open(
            registry_path,
            "r",
            encoding="utf-8"
        ) as f:

            registry = json.load(f)

    else:
        registry = {}

    already_exists = (
        mask_path.exists()
        or name in registry
    )

    if (
        already_exists
        and not overwrite_checkbox.value
    ):

        status.value = (
            "<span style='color:red;'>"
            f"'{name}' already exists. "
            "Check 'Overwrite' to replace it."
            "</span>"
        )

        return

    # Save compressed 3D boolean mask
    np.savez_compressed(
        mask_path,
        mask=custom_mask
    )

    # Record only slices that actually contain the custom region
    slice_indices = np.flatnonzero(
        custom_mask.any(
            axis=(0, 1)
        )
    ).tolist()

    atlas_plates = [
        slice_to_plate(i)
        for i in slice_indices
    ]

    base_region = locked_base_region

    if base_region == "None":
        base_region = None

    registry[name] = {

        "file": (
            f"data/custom_masks/"
            f"{name}.npz"
        ),

        "mask_key": "mask",

        "base_region": base_region,

        "atlas": "Allen-CCF-2020",

        "resolution_um": 25,

        "axis": 2,

        "plate_step_slices": 4,

        "slice_indices": slice_indices,

        "atlas_plates": atlas_plates,

        "voxel_count": int(
            custom_mask.sum()
        )
    }

    with open(
        registry_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            registry,
            f,
            indent=4
        )

    status.value = (
        "<span style='color:green;'>"
        f"✓ Saved custom region <b>{name}</b><br>"
        f"{len(atlas_plates)} atlas plate(s)<br>"
        f"{int(custom_mask.sum()):,} voxels<br>"
        f"Mask: {mask_path.name}"
        "</span>"
    )


save_button.on_click(
    save_custom_region
)

In [23]:
## 22. Build the interface
navigation_buttons = widgets.HBox(
    [
        prev_button,
        next_button
    ],
    layout=widgets.Layout(
        width="700px",
        justify_content="center"
    )
)

drawing_buttons = widgets.HBox(
    [
        draw_button,
        undo_point_button,
        finish_polygon_button,
        cancel_drawing_button
    ]
)

clear_buttons = widgets.HBox(
    [
        clear_plate_button,
        clear_all_button
    ]
)

save_controls = widgets.HBox(
    [
        save_button,
        overwrite_checkbox
    ]
)

In [24]:
## 23. Launch Custom Region Editor
custom_region_editor = widgets.VBox(
    [
        widgets.HTML(
            "<h2>Custom Region Editor</h2>"
            "<p>"
            "Draw an approved custom region across one or more "
            "Allen Atlas plates."
            "</p>"
        ),

        region_name_input,

        base_region_input,

        navigation_buttons,

        plate_slider,

        fig.canvas,

        drawing_buttons,

        clear_buttons,

        widgets.HTML("<hr>"),

        save_controls,

        status
    ],
    layout=widgets.Layout(
        width="900px"
    )
)

display(
    custom_region_editor
)

render_plate()

## How the saved mask will be used later

This notebook only creates and curates the custom mask.

The multiple ROI selector search combines:

1. Allen CCF regions
2. composite regions from `custom_regions.json`
3. curated masks from `custom_masks.json`

That will allow you to select `TH-Ant-MM` exactly like any other ROI without seeing the drawing tools.

<hr>

<div style="text-align: center; color: #777; font-size: 13px; margin-top: 25px;">
    Created by <b>Camila Vergara</b> ·
    <a href="https://github.com/macavero" target="_blank">
        GitHub
    </a>
</div>